## **Walkthrough of a Simple machine Learning Project cycle**

### **_Future Sales Prediction_**
This dataset provides information about the sales of a product and the advertising expenditures incurred by the business across various platforms. The following is a description of each column in the dataset:

- **TV**: Advertising cost spent in dollars for advertising on TV;
- **Radio**: Advertising cost spent in dollars for advertising on Radio;
- **Newspaper**: Advertising cost spent in dollars for advertising on Newspaper;
- **Sales**: Number of units sold;


In this dataset, the sales of the product are influenced by the advertising expenditures across different platforms. With this understanding of the dataset, the following section will guide you through the process of predicting future sales using machine learning techniques in Python.

### **Data Collection or Loading**

In [ ]:
# Import libraries and load the dataset
# You might need to load the data into the same directory

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

data = pd.read_csv("advertising.csv")
print(data.head())

### **EDA and Data Preprocessing**

In [ ]:
# Let’s have a look at whether this dataset contains any null values or not

print(data.isnull().sum())

In [ ]:
# Let's Visualise the data
# Visualise relationship between amount spendt on platforms and theie corresponding units sold

import plotly.express as px
import plotly.graph_objects as go

# TV

figure = px.scatter(data_frame = data, x="sales", # Changed 'Sales' to 'sales'
                    y="TV", size="TV", trendline="ols")
figure.show()

In [ ]:
figure = px.scatter(data_frame = data, x="sales", # Changed 'Sales' to 'sales' to match the column name
                    y="newspaper", size="newspaper", trendline="ols")
figure.show()

In [ ]:
# Radio
figure = px.scatter(data_frame = data, x="sales",
                    y="radio", size="radio", trendline="ols")
figure.show()

**Exercise**:
You have just done a repetitive task to visualise similar data  for the three platforms, there is an opportunity here to use reusable function.

Create a function to perform the plots by taking the column name as parameters

In [ ]:
# @title Create a function to visualise the data
# exercise

# def <call-it-a-suitable-name(camelCase)>:
  #the rest of the function goes here
import plotly.express as px

def visualizeSalesData(data, platform):
  """
  Visualizes the relationship between advertising expenditure on a given platform and sales.

  Args:
    data: The pandas DataFrame containing the advertising data.
    platform: The name of the advertising platform (e.g., "TV", "radio", "newspaper").

  Returns:
    None. Displays the generated scatter plot.
  """

  # Use the original platform name instead of converting to lowercase to match column name
  figure = px.scatter(data_frame=data,
                      x="sales",
                      y=platform,
                      size=platform,
                      trendline="ols")
  figure.show()

# Example usage:
visualizeSalesData(data, "TV") # Changed "tV" to "TV" to match the column name
visualizeSalesData(data, "radio")
visualizeSalesData(data, "newspaper")

### **Correlation Analysis**

In [ ]:
# let’s have a look at the correlation of all the columns with the sales column:

correlation = data.corr()

print(correlation["sales"].sort_values(ascending=False))

###**ML Model Training**

In [ ]:
# Before training, we need to create train and test Split

# Before training, we need to create train and test Split

x = np.array(data.drop(columns=["sales"])) # Use columns keyword instead of positional argument for axis
y = np.array(data["sales"])
xtrain, xtest, ytrain, ytest = train_test_split(x, y,
                                                test_size=0.2,
                                                random_state=42)

In [ ]:
# Now let’s train the model to predict future sales

model = LinearRegression() # this is a model we earlier imported from sklearn.linear_model
model.fit(xtrain, ytrain)

print(model.score(xtest, ytest))

###**Model Evaluation**

**Exercise**

Discuss methods to evaluate the performance of this ML trained model and implement one of the methods.

Is this model reliable to predict sales? Why not?

In [ ]:
# model evaluation
# Model evaluation using R-squared
r_squared = model.score(xtest, ytest)
print(f"R-squared: {r_squared}")

In [ ]:
# Other evaluation metrics: MAE, MSE and RMSE
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ypred = model.predict(xtest)

mae = mean_absolute_error(ytest, ypred)
mse = mean_squared_error(ytest, ypred)
rmse = np.sqrt(mse)
r2 = r2_score(ytest, ypred)

print(f"MAE: {mae}")
print(f"MSE: {mse}")
print(f"RMSE: {rmse}")
print(f"R-squared: {r2}")

In [ ]:
# Compare actual vs predicted sales on the test set
results = pd.DataFrame({"Actual": ytest, "Predicted": ypred})
print(results.head(10))

figure = px.scatter(data_frame = results, x="Actual", y="Predicted", trendline="ols",
                    title="Actual vs Predicted Sales")
figure.show()

In [ ]:
# Cross-validation: train and test the model 5 times on different splits
from sklearn.model_selection import cross_val_score

scores = cross_val_score(LinearRegression(), x, y, cv=5, scoring="r2")
print(f"R-squared for each fold: {scores}")
print(f"Average R-squared: {scores.mean()}")

**Answer – Methods to evaluate the model**

- **R-squared (R²):** shows how much of the variation in sales is explained by the advertising spend. 1 is perfect, 0 means the model explains nothing.
- **MAE (Mean Absolute Error):** the average difference between the real sales and the predicted sales, in units sold.
- **MSE (Mean Squared Error):** the average of the squared errors; it penalises big errors more.
- **RMSE (Root Mean Squared Error):** the square root of MSE, so it is back in units sold and easy to interpret.
- **Actual vs Predicted plot:** if the points are close to the diagonal line, the predictions are good.
- **Cross-validation:** the data is split several times (5 folds) and the model is tested on each part, to check that the result does not depend on one lucky split.

I implemented R², MAE, MSE, RMSE, the Actual vs Predicted plot and 5-fold cross-validation.

**Is this model reliable to predict sales?**

The results look good: R² is about 0.90 on the test set, and the cross-validation average is similar, so the model explains around 90% of the variation in sales, with an error (RMSE) of about 1.8 units.

However, it is **not fully reliable** for predicting future sales because:

1. **The dataset is very small** (only 200 rows), so the test set has only 40 rows. A few unusual cases can change the score a lot.
2. **Sales depend on many other factors** that are not in the data, such as price, season, competition, promotions or the economy.
3. **Linear regression assumes a straight-line relationship.** The TV plot shows the relationship is not perfectly linear (sales grow more slowly at high spend), and TV and radio may work together (interaction effect).
4. **Newspaper has a very weak correlation** with sales (0.23), so it adds little to the model.
5. **We can only trust predictions inside the range of the data.** If a company spends much more than the values in this dataset, the model may give wrong results.
6. The data has no dates, so we cannot check if the relationship changes over time.

So the model is a good starting point to understand which platform has more impact (TV, then radio), but it should be tested with more and newer data before being used for real business decisions.

### **Model Prediction**

In [ ]:
#features = [[TV, Radio, Newspaper]]

features = np.array([[230.1, 37.8, 69.2]])
print(model.predict(features))